# EXERCISE 2

In [14]:
import numpy as np
import pytest

### Task 2.1 Implement the DGEMM with matrices as NumPy array

In [15]:
def dgemm_numpy(A, B, C):
    """
    Perform DGEMM: C = C + A * B
    """
    N = A.shape[0]

    for i in range(N):
        for j in range(N):
            for k in range(N):
                C[i, j] += A[i, k] * B[k, j]

    return C

### Task 2.2 Using pytest develop a unit test for checking the correctness of your implementations.

In [16]:
from dgemm import dgemm_numpy

In [ ]:
import pytest
def test_dgemm_correctness():
    np.random.seed(0)
    N = 4

    A = np.random.rand(N, N).astype(np.float64)
    B = np.random.rand(N, N).astype(np.float64)
    C = np.random.rand(N, N).astype(np.float64)

    C_test = dgemm_numpy(A, B, C.copy())
    C_expected = C + A @ B

    assert np.allclose(C_test, C_expected)

@pytest.mark.parametrize("N", [1, 2, 5, 10])
def test_dgemm_multiple_sizes(N):
    np.random.seed(1)

    A = np.random.rand(N, N).astype(np.float64)
    B = np.random.rand(N, N).astype(np.float64)
    C = np.random.rand(N, N).astype(np.float64)

    C_test = dgemm_numpy(A, B, C.copy())
    C_expected = C + A @ B

    assert np.allclose(C_test, C_expected)



In [18]:
!python -m pytest -v test_dgemm.py

============================= test session starts ==============================
platform darwin -- Python 3.14.2, pytest-9.0.2, pluggy-1.6.0 -- /Users/mridulachandrika/Documents/Schoolwork/HPC-Intro/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/mridulachandrika/Documents/Schoolwork/HPC-Intro/Assignment2/EX2
collected 5 items                                                              

test_dgemm.py::test_dgemm_correctness PASSED                             [ 20%]
test_dgemm.py::test_dgemm_multiple_sizes[1] PASSED                       [ 40%]
test_dgemm.py::test_dgemm_multiple_sizes[2] PASSED                       [ 60%]
test_dgemm.py::test_dgemm_multiple_sizes[5] PASSED                       [ 80%]
test_dgemm.py::test_dgemm_multiple_sizes[10] PASSED                      [100%]

============================== 5 passed in 0.12s ===============================


The first test test_dgemm.py tests the correctness of the implementation by generating martices using random double-precision values and computing the equation: C = C + AB and checked whether the result matches with the expected values. The computed result is compared with NumPy’s built-in matrix multiplication (which relies on optimized BLAS routines) to ensure correctness.

The test_dgemm_multiple_sizes test extends this validation by repeating the correctness check for multiple matrix sizes, ensuring that the implementation works consistently across different problem dimensions.

### Task 2.3 Measure the execution time for each approach varying the matrix size. Report the average and error (std. deviation, min/max, or interval of confidence). Answer the question: how does the computational performance, e.g., the std, vary with increasing the size of the matrices, and why so?

In [19]:
import time

In [20]:
def time_dgemm(A, B, C, repeats=5):
    times = []

    for _ in range(repeats):
        C_copy = C.copy()
        start = time.perf_counter()
        dgemm_numpy(A, B, C_copy)
        end = time.perf_counter()
        times.append(end - start)

    return np.mean(times), np.std(times), min(times), max(times)


def main():
    np.random.seed(0)

    matrix_sizes = [32, 64, 96, 128]
    repeats = 5

    print(f"{'N':>6} | {'Mean (s)':>10} | {'Std (s)':>10} | {'Min (s)':>10} | {'Max (s)':>10}")
    print("-" * 60)

    for N in matrix_sizes:
        A = np.random.rand(N, N).astype(np.float64)
        B = np.random.rand(N, N).astype(np.float64)
        C = np.random.rand(N, N).astype(np.float64)

        mean_t, std_t, min_t, max_t = time_dgemm(A, B, C, repeats)

        print(f"{N:6d} | {mean_t:10.4f} | {std_t:10.4f} | {min_t:10.4f} | {max_t:10.4f}")

In [21]:
!python benchmark_dgemm.py

     N |   Mean (s) |    Std (s) |    Min (s) |    Max (s)
------------------------------------------------------------
    32 |     0.0144 |     0.0026 |     0.0115 |     0.0175
    64 |     0.0933 |     0.0046 |     0.0899 |     0.1019
    96 |     0.3083 |     0.0060 |     0.3023 |     0.3197
   128 |     0.7271 |     0.0141 |     0.7130 |     0.7512


The function time_dgemm() is used to measure the execution time of the DGEMM implementation. For each matrix size defined in the main function, the computation is repeated five times, and the mean execution time and standard deviation are calculated to analyze performance variability. The execution time increases rapidly as the matrix size grows, which is consistent with the 
𝑂(𝑁^3) computational complexity of the DGEMM algorithm. For example, doubling the matrix size from 64 to 128 increases the runtime by approximately a factor of eight, confirming the cubic scaling behavior. The absolute standard deviation increases with larger matrices, but the relative variability remains small.

### Task 2.4 Using the timing information and the number of operations for the DGEMM, calculate the FLOPS/s. How many operations are carried out in DGEMM with N as the matrices dimension? Hint: Think about the number of iterations completed in the loops and the number of flops per iteration. How do the FLOPS/s you measured compare to the theoretical peak of your processor

In [ ]:
def time_dgemm(A, B, C, repeats=5):
    times = []

    for _ in range(repeats):
        C_copy = C.copy()
        start = time.perf_counter()
        dgemm_numpy(A, B, C_copy)
        end = time.perf_counter()
        times.append(end - start)

    return np.mean(times), np.std(times)


def compute_flops(N):
    """
    DGEMM performs 2 * N^3 floating-point operations
    """
    return 2 * (N ** 3)


def main():
    np.random.seed(0)

    matrix_sizes = [32, 64, 96, 128]
    repeats = 5

    print(f"{'N':>6} | {'Mean Time (s)':>14} | {'Std (s)':>10} | {'FLOPs':>12} | {'FLOPs/s':>12}")
    print("-" * 75)

    for N in matrix_sizes:
        A = np.random.rand(N, N).astype(np.float64)
        B = np.random.rand(N, N).astype(np.float64)
        C = np.random.rand(N, N).astype(np.float64)

        mean_t, std_t = time_dgemm(A, B, C, repeats)

        flops = compute_flops(N)
        flops_per_sec = flops / mean_t

        print(
            f"{N:6d} | "
            f"{mean_t:14.4f} | "
            f"{std_t:10.4f} | "
            f"{flops:12.3e} | "
            f"{flops_per_sec:12.3e}"
        )

In [23]:
!python benchmark_dgemm_flops.py 

     N |  Mean Time (s) |    Std (s) |        FLOPs |      FLOPs/s
---------------------------------------------------------------------------
    32 |         0.0118 |     0.0003 |    6.554e+04 |    5.558e+06
    64 |         0.0931 |     0.0011 |    5.243e+05 |    5.634e+06
    96 |         0.3122 |     0.0041 |    1.769e+06 |    5.667e+06
   128 |         0.7410 |     0.0116 |    4.194e+06 |    5.661e+06


- For DGEMM, the total number of floating-point operations is 2𝑁^3, since each inner loop performs one multiplication and one addition. As the matrix size increases, the total FLOPs grow cubically, which explains the rapid increase in execution time.
- However, the measured FLOPs/s remains approximately constant at around 5.6×10^6 FLOPs/s for all tested matrix sizes. This indicates that the performance is bounded by Python’s execution overhead rather than the processor’s theoretical peak performance. Compared to the theoretical peak (assuming one operation per cycle at the CPU clock frequency), the achieved FLOPs/s is significantly lower, showing that pure Python implementations are not able to fully utilize the hardware capabilities.

### Task 2.5 Compare the performance results with the numpy matmul operation (that uses a BLAS library)

In [24]:
def time_python_dgemm(A, B, C, repeats=3):
    times = []

    for _ in range(repeats):
        C_copy = C.copy()
        start = time.perf_counter()
        dgemm_numpy(A, B, C_copy)
        end = time.perf_counter()
        times.append(end - start)

    return np.mean(times)


def time_numpy_dgemm(A, B, C, repeats=10):
    times = []

    for _ in range(repeats):
        C_copy = C.copy()
        start = time.perf_counter()
        C_copy += A @ B
        end = time.perf_counter()
        times.append(end - start)

    return np.mean(times)


def flops_dgemm(N):
    return 2 * (N ** 3)


def main():
    np.random.seed(0)

    matrix_sizes = [64, 128, 256]
    print(
        f"{'N':>6} | "
        f"{'Python DGEMM (s)':>18} | "
        f"{'NumPy BLAS (s)':>16} | "
        f"{'Speedup':>10} | "
        f"{'Python FLOPs/s':>16} | "
        f"{'BLAS FLOPs/s':>16}"
    )
    print("-" * 95)

    for N in matrix_sizes:
        A = np.random.rand(N, N).astype(np.float64)
        B = np.random.rand(N, N).astype(np.float64)
        C = np.random.rand(N, N).astype(np.float64)

        t_python = time_python_dgemm(A, B, C)
        t_numpy = time_numpy_dgemm(A, B, C)

        flops = flops_dgemm(N)

        flops_python = flops / t_python
        flops_numpy = flops / t_numpy
        speedup = t_python / t_numpy

        print(
            f"{N:6d} | "
            f"{t_python:18.4f} | "
            f"{t_numpy:16.6f} | "
            f"{speedup:10.1f} | "
            f"{flops_python:16.3e} | "
            f"{flops_numpy:16.3e}"
        )


In [25]:
!python compare_benchmark_dgemm_numpy.py 

     N |   Python DGEMM (s) |   NumPy BLAS (s) |    Speedup |   Python FLOPs/s |     BLAS FLOPs/s
-----------------------------------------------------------------------------------------------
    64 |             0.0913 |         0.000009 |     9866.8 |        5.742e+06 |        5.665e+10
   128 |             0.7452 |         0.000037 |    20263.3 |        5.629e+06 |        1.141e+11
   256 |             5.8422 |         0.000171 |    34106.5 |        5.743e+06 |        1.959e+11


- The results show a dramatic performance difference between the pure Python DGEMM implementation and NumPy’s BLAS-backed implementation. While the Python version achieves roughly 5.7×10^6 FLOPs/s consistently, the BLAS implementation reaches up to 10^11 FLOPs/s. This results in speedups ranging from approximately 9,800× to over 34,000× as the matrix size increases.
- These results clearly demonstrate why high-performance computing relies on optimized BLAS libraries rather than pure Python loops for computationally intensive linear algebra operations.
- The speedup increases with larger matrices because BLAS libraries are highly optimized, using techniques such as vectorization (SIMD), cache blocking, loop unrolling, and multi-threading to fully utilize the CPU hardware. In contrast, the Python implementation suffers from interpreter overhead and cannot exploit low-level hardware optimizations efficiently.